# 04 — Starting-Model Evaluation and Retrieval Diagnostics

## Objective
Measure the untouched language model with consistent role-aware prompts, a frozen validation partition, transparent class baselines, row-level predictions, and explicit error analysis. Separately verify retrieval-ranking formulas and optionally evaluate a reproducible **abstract-retrieval proxy** using real PubMedQA questions and abstracts.

A reported accuracy of 0.1667 on 30 questions (5/30 correct) is **not sufficient evidence** that the model is unsuitable. First inspect label frequencies, predicted-label frequencies, answer formatting, chat-template construction, truncation, and sampling uncertainty. Valid output format does not imply correct classification.

## Research design
- **Source:** `qiaojin/PubMedQA`, `pqa_labeled`; project-specific ID-based split from notebook 03.
- **Starting model:** `Qwen/Qwen2.5-0.5B-Instruct` by default, optionally the 1.5B variant.
- **Evaluation:** all validation examples by default (`EVAL_MAX_ITEMS=0`), with an optional fixed subset for a fast diagnostic run.
- **Generation:** tokenizer chat template, greedy decoding, raw answer retention, strict one-label parsing.
- **Baselines:** majority reference class and uniform 3-class chance reference (the latter is only a theoretical comparator, not a dataset-specific baseline).
- **Outcomes:** accuracy, macro-averaged F1, valid-format rate, confusion matrix, error table, and paired candidate metrics in later notebooks.

**Performance classification:** Results obtained with `RUN_MODEL_EVAL=1` are actual inference results only for the model, source revision, partition, and settings printed by this notebook. A sample of 30 is a diagnostic sample, not a reliable final benchmark.

### Workspace and run directory
All outputs are stored in an ignored local run directory, separate from the deployable source.

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


### PubMedQA schema, split integrity, and manifest verification
Validation uses the exact same dataset protocol as notebook 03. The original `context["contexts"]` sentences are retained, and an unexpected source fingerprint stops execution.

In [2]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter

DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def load_pubmedqa_source():
    """Load the labeled source, using a verified local Arrow file in offline mode."""
    from datasets import Dataset, config, load_dataset
    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        cache_root = Path(config.HF_DATASETS_CACHE)
        cached_files = list(cache_root.glob("qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
        if len(cached_files) != 1:
            raise FileNotFoundError("Expected one cached PubMedQA labeled Arrow file; found " + str(len(cached_files)))
        print("Source mode: existing Hugging Face Arrow cache")
        return Dataset.from_file(str(cached_files[0]))
    print("Source mode: Hugging Face load_dataset")
    return load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

### Prompt template and strict answer parsing
A Qwen instruction-tuned model expects its chat template. A plain string ending in `Answer:` can change behavior substantially. The parser accepts one answer label only; additional explanations are visible as invalid responses instead of silently counted as correct.

In [3]:
SYSTEM_INSTRUCTION = (
    "Answer the biomedical question using ONLY the supplied abstract evidence. "
    "Choose exactly one decision label: yes, no, or maybe. "
    "Return the label alone, without an explanation. "
    "Use maybe when the evidence does not establish yes or no."
)


def question_messages(row):
    """Use one consistent role-aware input protocol across model variants."""
    full_text = evidence_text(row)
    if not full_text:
        raise ValueError("Question has no evidence passage.")
    # Keep both the abstract introduction and conclusion; the conclusion often
    # carries the decision evidence. Apply the same limit in training/evaluation.
    if len(full_text) > 1600:
        text = full_text[:950] + "\n[Middle of abstract omitted for length]\n" + full_text[-600:]
    else:
        text = full_text
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"Question: {row['question']}\n\nAbstract evidence:\n{text}"},
    ]


def parse_decision(generated_text):
    """Parse an exact one-label answer; preserve raw outputs when parsing fails."""
    import re
    text = str(generated_text).strip()
    match = re.fullmatch(r"(?i)(yes|no|maybe)[.\s]*", text)
    return match.group(1).lower() if match else "invalid"

### Metrics and failure diagnostics
Accuracy measures exact decisions; macro-F1 weights each of the three reference classes equally. Valid-format rate is reported separately. A majority-class predictor provides a meaningful dataset-specific baseline.

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


def assess_predictions(frame):
    """Report output validity, class balance, and independent majority baseline."""
    if frame.empty:
        raise ValueError("No predictions provided for scoring.")
    y_true = frame["label"].astype(str).str.lower()
    y_pred = frame["prediction"].astype(str).str.lower()
    if not set(y_true).issubset(set(LABELS)):
        raise ValueError("Reference labels must be yes/no/maybe.")
    counts = y_true.value_counts()
    majority_label = counts.idxmax()
    summary = {
        "n": len(frame),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(LABELS), average="macro", zero_division=0)),
        "valid_format": float(y_pred.isin(LABELS).mean()),
        "majority_label": majority_label,
        "majority_accuracy": float((y_true == majority_label).mean()),
        "uniform_random_accuracy": 1 / len(LABELS),
    }
    return summary


def show_diagnostics(frame):
    """Provide evidence to distinguish formatting errors from actual classification errors."""
    metrics = assess_predictions(frame)
    print("Sample size:", metrics["n"])
    print("Reference distribution:", frame["label"].value_counts().to_dict())
    print("Predicted distribution:", frame["prediction"].value_counts().to_dict())
    print("Summary:", metrics)
    print("Confusion matrix (rows=reference, columns=prediction):")
    labels = list(LABELS) + ["invalid"]
    print(pd.DataFrame(confusion_matrix(frame["label"], frame["prediction"], labels=labels),
                       index=labels, columns=labels))
    mistakes = frame.loc[frame["prediction"] != frame["label"]]
    inspect_columns = [c for c in ("id", "label", "prediction", "raw", "input_token_count") if c in mistakes]
    if not mistakes.empty:
        print("Error samples (inspect generation and label handling):")
        display(mistakes[inspect_columns].head(8))
    if metrics["n"] < 100:
        print("CAUTION: A sample of fewer than 100 records gives an imprecise performance estimate.")
    if metrics["accuracy"] <= metrics["majority_accuracy"]:
        print("CAUTION: The model does not beat the validation majority-class baseline.")
    return metrics

### Metric unit test using invented predictions
These four rows are synthetic, and establish only that the scoring functions work.

In [5]:
demo = pd.DataFrame(
    [
        {"id": "d1", "label": "yes", "prediction": "yes", "raw": "yes"},
        {"id": "d2", "label": "no", "prediction": "no", "raw": "no"},
        {"id": "d3", "label": "maybe", "prediction": "invalid", "raw": "not enough data"},
        {"id": "d4", "label": "yes", "prediction": "maybe", "raw": "maybe"},
    ]
)
metrics = show_diagnostics(demo)
assert metrics["accuracy"] == 0.5
assert parse_decision("No. ") == "no"
assert parse_decision("The answer is yes") == "invalid"

Sample size: 4
Reference distribution: {'yes': 2, 'no': 1, 'maybe': 1}
Predicted distribution: {'yes': 1, 'no': 1, 'invalid': 1, 'maybe': 1}
Summary: {'n': 4, 'accuracy': 0.5, 'macro_f1': 0.5555555555555555, 'valid_format': 0.75, 'majority_label': 'yes', 'majority_accuracy': 0.5, 'uniform_random_accuracy': 0.3333333333333333}
Confusion matrix (rows=reference, columns=prediction):
         yes  no  maybe  invalid
yes        1   0      1        0
no         0   1      0        0
maybe      0   0      0        1
invalid    0   0      0        0
Error samples (inspect generation and label handling):


,id,label,prediction,raw
2,d3,maybe,invalid,not enough data
3,d4,yes,maybe,maybe


CAUTION: A sample of fewer than 100 records gives an imprecise performance estimate.
CAUTION: The model does not beat the validation majority-class baseline.


### Deterministic decoding and full prediction capture
For traceability, the model output is saved alongside the publication ID, reference class, and input token count. Sampling flags embedded in the downloaded model generation configuration are overridden with a deterministic generation configuration.

In [6]:
# Use the tokenizer's instruction-chat format: raw plain-text prompts are not equivalent.
# Extract newly generated tokens only, and preserve the exact decoded text for auditing.
def generate_answer(model, tokenizer, row, max_input_tokens=768, max_output_tokens=8):
    import torch
    from transformers import GenerationConfig

    if tokenizer.chat_template is None:
        raise ValueError("Model tokenizer does not provide the expected chat template.")
    chat_prompt = tokenizer.apply_chat_template(
        question_messages(row), tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(
        chat_prompt,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=False,  # Chat template already contains special tokens.
    )
    input_length = int(encoded["input_ids"].shape[-1])
    if input_length > max_input_tokens:
        raise ValueError(
            f"Paper {paper_id(row)} exceeds the input token budget "
            f"({input_length} > {max_input_tokens}); revise the shared evidence budget. "
            "Do not truncate the assistant generation marker."
        )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    # Ignore sampling values embedded in a model repository's generation_config.json.
    # A freshly constructed config keeps this run deterministic and avoids warnings
    # about sampling-only flags when do_sample=False.
    config = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_output_tokens,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    with torch.inference_mode():
        sequence = model.generate(**encoded, generation_config=config)
    response = tokenizer.decode(sequence[0, input_length:], skip_special_tokens=True)
    return {"raw": response, "prediction": parse_decision(response), "input_token_count": input_length}

### Actual unmodified-model validation
Set `RUN_MODEL_EVAL=1` to run the original-data and model evaluation. With `HF_DATASETS_OFFLINE=1`, a single existing PubMedQA Arrow cache is read directly; otherwise Hugging Face Datasets loads the source. The source fingerprint must match the manifest established in notebook 03. `EVAL_MAX_ITEMS=0` evaluates the full validation partition; a smaller value is intended only for debugging and is recorded as such.

In [7]:
if os.getenv("RUN_MODEL_EVAL", "0") == "1":
    import gc
    import time
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    max_items = int(os.getenv("EVAL_MAX_ITEMS", "0"))
    source = load_pubmedqa_source()
    partitions = prepare_partitions(source, create_manifest=False)
    validation = sorted(partitions["validation"], key=paper_id)
    if max_items > 0:
        validation = validation[:max_items]
        print("DIAGNOSTIC SUBSET: the following scores are not a full-validation benchmark.")

    print("Model:", model_id, "Questions:", len(validation))
    print("Reference class frequencies:", dict(Counter(r["final_decision"] for r in validation)))
    torch.manual_seed(SEED)
    run_start = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
    model.eval()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    inference_start = time.perf_counter()

    records = []
    for index, record in enumerate(validation, start=1):
        generated = generate_answer(model, tokenizer, record)
        records.append({
            "id": paper_id(record),
            "label": str(record["final_decision"]).lower(),
            "question": str(record["question"]),
            **generated,
            "model_id": model_id,
        })
        if index % 20 == 0 or index == len(validation):
            print(f"Evaluated {index}/{len(validation)} questions")

    inference_seconds = time.perf_counter() - inference_start
    elapsed_seconds = time.perf_counter() - run_start
    peak_gpu_gib = (torch.cuda.max_memory_allocated() / 2**30) if torch.cuda.is_available() else None
    predictions = pd.DataFrame(records)
    print("Actual starting-model evaluation:")
    show_diagnostics(predictions)
    output = RUNS / "predictions_base_validation.csv"
    predictions.to_csv(output, index=False)
    protocol = {
        "model_id": model_id,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "device": str(model.device),
        "model_dtype": str(model.dtype),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "seed": SEED,
        "source_partition": "validation",
        "dataset_fingerprint": json.loads(MANIFEST.read_text())["fingerprint"],
        "evaluated_ids": predictions["id"].tolist(),
        "decoding": "chat_template/greedy/max_new_tokens=8",
        "max_items": max_items,
        "model_variant": "base",
        "inference_seconds": round(inference_seconds, 3),
        "elapsed_seconds": round(elapsed_seconds, 3),
        "peak_gpu_allocated_gib": round(peak_gpu_gib, 3) if peak_gpu_gib is not None else None,
    }
    (RUNS / "predictions_base_validation_manifest.json").write_text(
        json.dumps(protocol, indent=2) + "\n", encoding="utf-8"
    )
    print("Saved row-level predictions and protocol:", output)
    print("Inference seconds:", round(inference_seconds, 3), "Peak GPU allocated GiB:", protocol["peak_gpu_allocated_gib"])
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Model evaluation skipped. Set RUN_MODEL_EVAL=1 to run genuine validation.")

Source mode: existing Hugging Face Arrow cache
train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}
Model: Qwen/Qwen2.5-0.5B-Instruct Questions: 121
Reference class frequencies: {'yes': 64, 'no': 40, 'maybe': 17}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Evaluated 20/121 questions
Evaluated 40/121 questions
Evaluated 60/121 questions
Evaluated 80/121 questions
Evaluated 100/121 questions
Evaluated 120/121 questions
Evaluated 121/121 questions
Actual starting-model evaluation:
Sample size: 121
Reference distribution: {'yes': 64, 'no': 40, 'maybe': 17}
Predicted distribution: {'maybe': 119, 'no': 2}
Summary: {'n': 121, 'accuracy': 0.15702479338842976, 'macro_f1': 0.11507936507936507, 'valid_format': 1.0, 'majority_label': 'yes', 'majority_accuracy': 0.5289256198347108, 'uniform_random_accuracy': 0.3333333333333333}
Confusion matrix (rows=reference, columns=prediction):
         yes  no  maybe  invalid
yes        0   0     64        0
no         0   2     38        0
maybe      0   0     17        0
invalid    0   0      0        0
Error samples (inspect generation and label handling):


,id,label,prediction,raw,input_token_count
0,10135926,yes,maybe,maybe,329
1,10548670,yes,maybe,maybe,432
2,10798511,no,maybe,maybe,251
3,11138995,no,maybe,maybe,338
5,11566686,yes,maybe,maybe,332
6,11713724,no,maybe,maybe,433
7,11926574,no,maybe,maybe,436
8,11943048,yes,maybe,maybe,489


CAUTION: The model does not beat the validation majority-class baseline.
Saved row-level predictions and protocol: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\predictions_base_validation.csv
Inference seconds: 5.59 Peak GPU allocated GiB: 0.956


### Results

The pinned Qwen2.5-0.5B-Instruct base answered 19/121 (15.7%) validation questions correctly, with macro-F1 0.115 and 121/121 valid one-label outputs. It predicted maybe for 119/121 cases. The always-yes reference scored 64/121 (52.9%).

### Interpretation

The shared chat-template/greedy protocol establishes a particularly weak starting-model baseline. Correct formatting did not imply correct answers.

### Limitations

The 121 cases are a project-specific validation split, not PubMedQA's official benchmark. The earlier 30-case screenshot was a separate diagnostic and is not substituted for this result.

### Key Observations

Later gains should be compared on these same IDs and against the majority rule, with classwise errors retained.


### How to interpret a low accuracy
- **Check reference balance:** compare against the majority-class accuracy, not only the uniform 1/3 theoretical reference.
- **Check predicted balance:** all-yes or all-maybe answers can produce a deceptively high valid-format score and poor accuracy.
- **Inspect raw responses:** a high invalid fraction signals formatting or decoding problems; a low invalid fraction with poor accuracy signals classification problems.
- **Inspect context coverage:** printed input lengths and truncation notices indicate whether evidence is being dropped.
- **Match checkpoints:** Qwen2.5-0.5B-Instruct and Qwen2.5-1.5B-Instruct are different baselines; record their model IDs.
- **Estimate uncertainty:** 30 examples are too few for a strong performance claim. Increase evaluation coverage and repeat robustness checks.

## Retrieval ranking is a separate experiment
Retrieval evaluates whether the relevant *document* was found. Answer accuracy evaluates the *decision* returned by a language model. These cannot be treated as the same measurement.

For query $q$, with relevant set $R_q$ and top-$k$ retrieved list $D_q^{(k)}$:

$$
\mathrm{Recall@}k(q) = \frac{|R_q \cap D_q^{(k)}|}{|R_q|},
\qquad
\mathrm{Precision@}k(q) = \frac{|R_q \cap D_q^{(k)}|}{k}.
$$

$$
\mathrm{MRR@}k = \frac{1}{|Q|} \sum_{q \in Q}
\frac{1}{\mathrm{rank}_q},
$$
where the reciprocal rank is zero when the top-$k$ list contains no relevant document.

### Unit-check of retrieval formulas (illustrative records)
Query A has two relevant papers, both retrieved; the first occurs at rank 2. Query B has one relevant paper and none retrieved. These numbers verify the metric implementation, not a deployed search engine.

In [8]:
from statistics import mean


def retrieval_metrics(relevant_ids, retrieved_ids, k):
    """Score a ranked retrieval list against independently supplied relevant IDs."""
    if not relevant_ids:
        raise ValueError("Relevance judgments are needed to compute recall.")
    if k <= 0:
        raise ValueError("k must be positive.")
    top_k = list(retrieved_ids[:k])
    relevant = set(relevant_ids)
    hits = relevant.intersection(top_k)
    first_rank = next((rank for rank, doc_id in enumerate(top_k, 1) if doc_id in relevant), None)
    return {
        "recall_at_k": len(hits) / len(relevant),
        "precision_at_k": len(hits) / k,
        "reciprocal_rank_at_k": 1.0 / first_rank if first_rank is not None else 0.0,
    }

illustrative_queries = [
    {"query": "A", "relevant": {"paper_1", "paper_3"}, "retrieved": ["paper_2", "paper_1", "paper_3"]},
    {"query": "B", "relevant": {"paper_7"}, "retrieved": ["paper_2", "paper_8", "paper_4"]},
]
metrics_by_query = [retrieval_metrics(q["relevant"], q["retrieved"], 3) for q in illustrative_queries]
for query, metrics in zip(illustrative_queries, metrics_by_query):
    print(query["query"], metrics)
print("Illustrative mean Recall@3:", mean(m["recall_at_k"] for m in metrics_by_query))
assert metrics_by_query[0]["recall_at_k"] == 1.0
assert metrics_by_query[0]["precision_at_k"] == 2 / 3
assert metrics_by_query[0]["reciprocal_rank_at_k"] == 0.5
assert metrics_by_query[1]["reciprocal_rank_at_k"] == 0.0

A {'recall_at_k': 1.0, 'precision_at_k': 0.6666666666666666, 'reciprocal_rank_at_k': 0.5}
B {'recall_at_k': 0.0, 'precision_at_k': 0.0, 'reciprocal_rank_at_k': 0.0}
Illustrative mean Recall@3: 0.5


### Optional real-source retrieval benchmark: TF–IDF (Term Frequency–Inverse Document Frequency)
This is a **retrieval proxy** rather than a production Retrieval-Augmented Generation (RAG) pipeline:

1. Index original abstracts from the validation portion of `pqa_labeled`.
2. Use each corresponding original research question as a query.
3. Treat the question's own publication abstract as the known matching document.
4. Rank all indexed abstracts by TF–IDF cosine similarity, then calculate Recall@3, Precision@3, and Mean Reciprocal Rank (MRR@3).

The known matching abstract is a **proxy relevance label** derived from PubMedQA pairing; other relevant papers may exist. These metrics characterize this paired-document benchmark, not general RAG retrieval quality. No generative-answer groundedness is measured here. Set `RUN_RETRIEVAL_EVAL=1` to run this small genuine source-backed comparison.

In [9]:
if os.getenv("RUN_RETRIEVAL_EVAL", "0") == "1":
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    source = load_pubmedqa_source()
    partitions = prepare_partitions(source, create_manifest=False)
    subset = sorted(partitions["validation"], key=paper_id)
    corpus_ids = [paper_id(row) for row in subset]
    documents = [evidence_text(row) for row in subset]
    queries = [str(row["question"]) for row in subset]

    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
    document_vectors = vectorizer.fit_transform(documents)
    query_vectors = vectorizer.transform(queries)
    similarities = cosine_similarity(query_vectors, document_vectors)

    retrieval_rows = []
    for i, scores in enumerate(similarities):
        order = np.argsort(-scores, kind="stable")[:3]
        retrieved = [corpus_ids[j] for j in order]
        metrics = retrieval_metrics({corpus_ids[i]}, retrieved, k=3)
        retrieval_rows.append({"query_id": corpus_ids[i], "retrieved_ids": json.dumps(retrieved), **metrics})

    retrieval_frame = pd.DataFrame(retrieval_rows)
    display(retrieval_frame.head())
    print("Original-document matching proxy, number of queries:", len(retrieval_frame))
    print(retrieval_frame[["recall_at_k", "precision_at_k", "reciprocal_rank_at_k"]].mean())
    path = RUNS / "pubmedqa_tfidf_retrieval_proxy_validation.csv"
    retrieval_frame.to_csv(path, index=False)
    print("Saved:", path)
else:
    print("Source-backed retrieval skipped. Set RUN_RETRIEVAL_EVAL=1 after preparing PubMedQA data.")

Source mode: existing Hugging Face Arrow cache
train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}


,query_id,retrieved_ids,recall_at_k,precision_at_k,reciprocal_rank_at_k
0,10135926,"[""10135926"", ""22427593"", ""11713724""]",1.0,0.333333,1.0
1,10548670,"[""10548670"", ""19142546"", ""19155657""]",1.0,0.333333,1.0
2,10798511,"[""10798511"", ""18847643"", ""20871246""]",1.0,0.333333,1.0
3,11138995,"[""11138995"", ""12920330"", ""21276532""]",1.0,0.333333,1.0
4,11458136,"[""11458136"", ""20629769"", ""12238307""]",1.0,0.333333,1.0


Original-document matching proxy, number of queries: 121
recall_at_k             0.991736
precision_at_k          0.330579
reciprocal_rank_at_k    0.987603
dtype: float64
Saved: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\pubmedqa_tfidf_retrieval_proxy_validation.csv


### Results — Retrieval proxy

Term frequency–inverse document frequency matching recovered the originating abstract for validation questions with Recall@3 0.9917, Precision@3 0.3306, and mean reciprocal rank 0.9876.

### Interpretation

The proxy tests question-to-originating-document recovery in this constructed corpus.

### Limitations

The paired abstract is not an independently judged exhaustive relevance label. These figures do not establish a live vector index, answer grounding, or generated-answer correctness.

### Key Observations

Retrieval ranking and downstream biomedical answer quality must be evaluated separately.


## Key observations
The completed full-validation run had 100% answer-format validity but only 15.7% accuracy; the earlier 30-question figures were a separate diagnostic. Format compliance alone is not reliable evidence of correct biomedical reasoning. Save the raw strings and confusion matrix before changing model checkpoints or training methods.

Illustrative Query A/B retrieval metrics are correct but do not evaluate a live index. The source-backed TF–IDF exercise provides a more meaningful baseline for paired-document matching, while still explicitly limiting the relevance-label interpretation.

## Q&A
**Question:** Does 16.7% validation accuracy prove that the base model is worse than random?  
**Answer:** No. The reference distribution, sample size, uncertainty, prompt format, output parser, and majority predictor must be checked. Uniform random expectation is 1/3 only under equally likely guesses; it is not a universally appropriate performance baseline.

**Question:** Can Recall@3 of 1.0 guarantee a correct answer?  
**Answer:** No. A generator can hallucinate or misinterpret even perfectly retrieved evidence. Retrieval relevance and generated-answer correctness must be evaluated separately.